In [ ]:
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_sharegpt

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

# Load from HF Hub
HF_DATASET = "clemsail/mascarade-embedded-dataset"
dataset = load_dataset(HF_DATASET, split="train")
# Or local: dataset = load_dataset("json", data_files={"train": "embedded_chat.jsonl"}, split="train")

dataset = standardize_sharegpt(dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False
    ) for convo in convos]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset: {len(dataset)} examples")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=3407,
    ),
)

trainer_stats = trainer.train()
print(f"Training time: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Write bare-metal ARM Cortex-M4 SysTick timer init for 1ms tick",
    "Configure ESP32 SPI master at 40MHz with DMA for W25Q128 flash",
    "Write RISC-V assembly for a GPIO toggle on SiFive HiFive1",
    "Implement STM32 DMA + ADC continuous conversion with LL drivers",
    "Write a Raspberry Pi bare-metal UART driver in C",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": "You are an expert embedded systems engineer specializing in ARM Cortex-M, ESP32/ESP-IDF, and RISC-V."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.3)
    print(f"\n{'='*60}\nQ: {prompt}\n{'='*60}")
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
HF_USERNAME = "clemsail"
MODEL_NAME = "mascarade-embedded-q4km"

model.push_to_hub_gguf(
    f"{HF_USERNAME}/{MODEL_NAME}",
    tokenizer,
    quantization_method="q4_k_m",
)
print(f"Model pushed to https://huggingface.co/{HF_USERNAME}/{MODEL_NAME}")

## Deploy locally

```bash
# On your machine:
./deploy_model.sh embedded clemsail/mascarade-embedded-q4km
# Or manually:
huggingface-cli download clemsail/mascarade-embedded-q4km --local-dir ./models/embedded/
ollama create mascarade-embedded -f modelfiles/Modelfile.embedded
```